# 📊 EDA & Feature Discovery Notebook for Expense Categorization
**Target Task:** Categorize transaction expense descriptions into 10 categories.  
**Key Goal:** Discover new feature representations across 5 key dimensions:
1. **Financial & Cents Patterns** (`cents`, `is_round_dollar`, `is_99_cents`, `amount_bin`, `log_amount`).
2. **Text Structural & Regex Signals** (`has_store_num`, `has_state_code`, `has_hash`, `has_star`, `has_dot_com`, `digit_ratio`).
3. **Merchant Root & Frequency** (`merchant_root`, `merchant_freq`).
4. **Temporal & Recurring Dynamics** (`is_payday`, `day_of_week`, `dow_sin/cos`, `week_of_month`).
5. **Hybrid Text Embeddings** (TF-IDF Word (1-3 ngrams) + TF-IDF Char (3-5 ngrams)).


In [ ]:
import re
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
import lightgbm as lgb
from sklearn.metrics import accuracy_score, f1_score

train = pd.read_csv('/kaggle/input/competitions/aurora-gate-expense-categorization-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/aurora-gate-expense-categorization-challenge/test.csv')
all_df = pd.concat([train, test], ignore_index=True)

print(f"Train shape: {train.shape}, Test shape: {test.shape}")
train['category'].value_counts()


In [ ]:
state_pattern = r'\b(AL|AK|AZ|AR|CA|CO|CT|DE|FL|GA|HI|ID|IL|IN|IA|KS|KY|LA|ME|MD|MA|MI|MN|MS|MO|MT|NE|NV|NH|NJ|NM|NY|NC|ND|OH|OK|OR|PA|RI|SC|SD|TN|TX|UT|VT|VA|WA|WV|WI|WY|DC)\b'

def extract_merchant_root(desc):
    if not isinstance(desc, str): return ''
    s = desc.upper()
    s = re.sub(state_pattern + r'$', '', s).strip()
    s = re.sub(r'#?\d+', '', s).strip()
    s = re.sub(r'\s+', ' ', s).strip()
    return s

all_df['merchant_root'] = all_df['description'].apply(extract_merchant_root)
merchant_freq_map = all_df['merchant_root'].value_counts().to_dict()

def build_features(df):
    feats = pd.DataFrame(index=df.index)
    desc = df['description'].astype(str)
    
    # Financial
    feats['amount'] = df['amount']
    feats['log_amount'] = np.log1p(df['amount'])
    cents = (df['amount'] * 100 % 100).round().astype(int)
    feats['cents'] = cents
    feats['is_round_dollar'] = (cents == 0).astype(int)
    feats['is_99_cents'] = (cents == 99).astype(int)
    feats['is_95_cents'] = (cents == 95).astype(int)
    feats['is_50_cents'] = (cents == 50).astype(int)
    feats['amount_mod_5'] = ((cents == 0) & (df['amount'].astype(int) % 5 == 0)).astype(int)
    feats['amount_mod_10'] = ((cents == 0) & (df['amount'].astype(int) % 10 == 0)).astype(int)
    feats['amount_bin'] = pd.qcut(all_df['amount'], q=10, labels=False, duplicates='drop').loc[df.index]
    
    # Text Structural
    feats['char_len'] = desc.apply(len)
    feats['word_len'] = desc.apply(lambda x: len(x.split()))
    feats['digit_count'] = desc.apply(lambda x: sum(c.isdigit() for c in x))
    feats['digit_ratio'] = feats['digit_count'] / (feats['char_len'] + 1e-5)
    feats['uppercase_count'] = desc.apply(lambda x: sum(c.isupper() for c in x))
    feats['uppercase_ratio'] = feats['uppercase_count'] / (feats['char_len'] + 1e-5)
    
    feats['has_hash'] = desc.str.contains('#', regex=False).astype(int)
    feats['has_star'] = desc.str.contains(r'\*', regex=True).astype(int)
    feats['has_slash'] = desc.str.contains('/', regex=False).astype(int)
    feats['has_dot_com'] = desc.str.lower().str.contains(r'\.com', regex=True).astype(int)
    feats['has_store_num'] = desc.str.contains(r'#?\d{3,}', regex=True).astype(int)
    feats['has_state_code'] = desc.str.contains(state_pattern, regex=True).astype(int)
    
    # Merchant frequency
    m_roots = desc.apply(extract_merchant_root)
    feats['merchant_freq'] = m_roots.map(merchant_freq_map).fillna(0)
    
    # Temporal
    dt = pd.to_datetime(df['date'])
    feats['year'] = dt.dt.year
    feats['month'] = dt.dt.month
    feats['day'] = dt.dt.day
    feats['dayofweek'] = dt.dt.dayofweek
    feats['is_weekend'] = (feats['dayofweek'] >= 5).astype(int)
    feats['is_month_start'] = dt.dt.is_month_start.astype(int)
    feats['is_month_end'] = dt.dt.is_month_end.astype(int)
    feats['is_payday'] = feats['day'].isin([1, 2, 14, 15, 16, 28, 29, 30, 31]).astype(int)
    feats['week_of_month'] = (feats['day'] - 1) // 7 + 1
    
    feats['month_sin'] = np.sin(2 * np.pi * feats['month'] / 12)
    feats['month_cos'] = np.cos(2 * np.pi * feats['month'] / 12)
    feats['day_sin'] = np.sin(2 * np.pi * feats['day'] / 31)
    feats['day_cos'] = np.cos(2 * np.pi * feats['day'] / 31)
    feats['dow_sin'] = np.sin(2 * np.pi * feats['dayofweek'] / 7)
    feats['dow_cos'] = np.cos(2 * np.pi * feats['dayofweek'] / 7)
    
    date_counts = all_df['date'].value_counts()
    feats['date_trans_count'] = df['date'].map(date_counts).fillna(0)
    
    return feats

train_feats = build_features(train)
train_feats['category'] = train['category']
print("Engineered feature preview:")
train_feats.head()


In [ ]:
# EDA: Means of structural binary features by category
binary_cols = ['has_store_num', 'has_state_code', 'is_round_dollar', 'is_payday', 'has_star', 'has_hash', 'has_dot_com']
train_feats.groupby('category')[binary_cols].mean().round(4)


In [ ]:
def clean_text(s):
    if not isinstance(s, str): return ''
    s = s.lower()
    s = re.sub(r'[#*\-_/\\|@&%$]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

train_desc_clean = train['description'].apply(clean_text)

word_vec = TfidfVectorizer(ngram_range=(1, 3), max_features=2500, min_df=2, sublinear_tf=True)
X_word = word_vec.fit_transform(train_desc_clean)

char_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=3500, min_df=2, sublinear_tf=True)
X_char = char_vec.fit_transform(train_desc_clean)

num_cols = [c for c in train_feats.columns if c != 'category']
X_num = train_feats[num_cols].values

X_all = hstack([X_num, X_word, X_char]).tocsr()

le = LabelEncoder()
y_train = le.fit_transform(train['category'])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train))
feature_names = num_cols + [f"word_{i}" for i in range(X_word.shape[1])] + [f"char_{i}" for i in range(X_char.shape[1])]
feature_importances = np.zeros(len(feature_names))

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_all, y_train)):
    clf = lgb.LGBMClassifier(
        n_estimators=350, learning_rate=0.04, num_leaves=31, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, class_weight='balanced',
        random_state=42 + fold, n_jobs=4, verbose=-1
    )
    clf.fit(X_all[trn_idx], y_train[trn_idx])
    preds = clf.predict(X_all[val_idx])
    oof_preds[val_idx] = preds
    feature_importances += clf.feature_importances_ / skf.n_splits

cv_acc = accuracy_score(y_train, oof_preds)
cv_f1 = f1_score(y_train, oof_preds, average='macro')
print(f"=== 5-Fold Stratified CV Accuracy: {cv_acc:.4f} | Macro F1: {cv_f1:.4f} ===")

fi_df = pd.DataFrame({'feature': feature_names, 'importance': feature_importances}).sort_values('importance', ascending=False)
tabular_fi = fi_df[fi_df['feature'].isin(num_cols)].sort_values('importance', ascending=False)
print("Top Tabular & Feature Signals:")
print(tabular_fi.head(20))
